# Support vector regression (SVR)

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
SVR draws a **tube** of width $\varepsilon$ around the prediction curve. Errors inside the tube are **free**; only points outside the tube are penalized. The goal: the flattest function that keeps most points inside the tube.

## The ε-insensitive loss
$$L_\varepsilon(y,\hat y)=\max(0,\ \vert y-\hat y\vert-\varepsilon)$$

> [Squared error punishes every error; the SVR loss is zero up to ε and then grows linearly (more robust to outliers).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/svr_loss.png)

*Squared error punishes every error; the SVR loss is zero up to ε and then grows linearly (more robust to outliers).*

The optimization problem:

$$\min_{\mathbf w,b}\ \frac12\Vert\mathbf w\Vert^2+C\sum_i L_\varepsilon(y_i,\hat y_i)$$

- $\frac12\Vert\mathbf w\Vert^2$ keeps the function flat.
- $C$ sets how much points outside the tube cost.

## Support vectors
Only points on the edge of the tube or outside it shape the answer; these are the **support vectors**. Points inside the tube could be removed without changing the model.

> [With a larger ε, the tube is wider and there are fewer support vectors (red circles).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/svr_eps.png)

*With a larger ε, the tube is wider and there are fewer support vectors (red circles).*

## The kernel trick
With a kernel, SVR learns non-linear relations without building new features. The RBF kernel measures how similar two points are:

$$K(x,x')=\exp(-\gamma\Vert x-x'\Vert^2)$$

and the prediction is $\hat y(x)=\sum_{i\in SV}\beta_iK(x_i,x)+b$.

[[lab:reg_svr]]

## Three hyperparameters
| Hyperparameter | Small | Large |
|---|---|---|
| `C` | Flat function, more error (underfitting) | Chases the points (overfitting) |
| `epsilon` | Narrow tube, many support vectors | Wide tube, simpler model |
| `gamma` (RBF) | Each point affects a wide area; smooth | Local effect; wiggly |

> SVR is very sensitive to scale: always apply `StandardScaler` to X. It is slow beyond tens of thousands of samples.



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

X, y = load_diabetes(return_X_y=True)
pipe = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
grid = GridSearchCV(pipe, {"svr__C": [1, 10, 100, 300], "svr__epsilon": [1, 10, 30], "svr__gamma": ["scale", 0.01, 0.1]}, cv=5, scoring="r2")
grid.fit(X, y)
print("Best:", grid.best_params_, " R2 =", round(grid.best_score_, 3))
best = grid.best_estimator_[-1]
print("Support vectors:", len(best.support_), "of", len(X))



| Pros | Cons |
|---|---|
| Robust to outliers (compared with MSE) | Slow on big data ($O(n^2)$ to $O(n^3)$) |
| Non-linear relations with kernels | Three sensitive hyperparameters |
| Excellent on small and medium data | Hard to interpret |

### Exercise

On a sine data set, print how epsilon changes the number of support vectors.

In [ ]:
# Your answer here